In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["VECLIB_MAXIMUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [3]:
import pandas as pd
import numpy as np

DATA_DIR = '../data/'

train = pd.read_csv(DATA_DIR + 'GLC25_PA_metadata_train.csv')
test  = pd.read_csv(DATA_DIR + 'GLC25_PA_metadata_test.csv')

print('Train shape:', train.shape)
print('Test  shape:', test.shape)

Train shape: (1483637, 9)
Test  shape: (14784, 8)


In [4]:
print(train.dtypes)
print(test.dtypes)

lon                  float64
lat                  float64
year                   int64
geoUncertaintyInM    float64
areaInM2             float64
region                object
country               object
speciesId            float64
surveyId               int64
dtype: object
lon                  float64
lat                  float64
year                   int64
geoUncertaintyInM    float64
areaInM2             float64
region                object
country               object
surveyId               int64
dtype: object


In [5]:
print('=== Valeurs manquantes TRAIN ===')
print(train.isnull().sum())
print('\n=== Valeurs manquantes TEST ===')
print(test.isnull().sum())

=== Valeurs manquantes TRAIN ===
lon                       0
lat                       0
year                      0
geoUncertaintyInM     12496
areaInM2             183272
region                    0
country                   0
speciesId                 0
surveyId                  0
dtype: int64

=== Valeurs manquantes TEST ===
lon                    0
lat                    0
year                   0
geoUncertaintyInM    840
areaInM2             644
region                 0
country                0
surveyId               0
dtype: int64


In [6]:
train['lat_lon'] = train['lat'] * train['lon']
train['lat2']= train['lat'] ** 2
train['lon2'] = train['lon'] ** 2

In [7]:
print(train.corr(numeric_only=True)["speciesId"].sort_values(ascending=False))

speciesId            1.000000
lat2                 0.015254
lat                  0.014907
lat_lon              0.005264
year                 0.003626
lon                  0.003009
geoUncertaintyInM    0.001994
surveyId             0.000730
lon2                 0.000599
areaInM2            -0.017997
Name: speciesId, dtype: float64


1. Préparation des données

In [8]:
features = ['lon', 'lat','lat_lon', 'lat2', 'year']

X = train.groupby('surveyId')[features + ['region', 'country']].first()
X = pd.get_dummies(X, columns=['region', 'country'])


In [9]:
TOP_K_SPECIES = 350

top_species = train['speciesId'].value_counts().head(TOP_K_SPECIES).index

train_filtered = train[train['speciesId'].isin(top_species)]

Y = (
    train_filtered
    .assign(value=1)
    .pivot_table(index='surveyId', columns='speciesId', values='value', fill_value=0)
)

X = X.loc[Y.index]

In [10]:
from xgboost import XGBClassifier
import numpy as np

# nettoyage
X = X.astype(float)
X = X.replace([np.inf, -np.inf], 0)
X = X.fillna(0)

models = {}

for species in Y.columns:
    y = Y[species].astype(int)
    
    # skip espèces inutiles
    if y.nunique() < 2:
        continue
    
    print(f"Training species {species}")
    
    model = XGBClassifier(
        n_estimators=3,   # ↓ pour commencer (rapide)
        max_depth=5,
        learning_rate=0.1,
        tree_method='hist',
        n_jobs=8
    )
    
    model.fit(X, y)
    models[species] = model

Training species 53.0
Training species 96.0
Training species 129.0
Training species 146.0
Training species 167.0
Training species 249.0
Training species 254.0
Training species 262.0
Training species 300.0
Training species 340.0
Training species 391.0
Training species 394.0
Training species 423.0
Training species 433.0
Training species 469.0
Training species 478.0
Training species 540.0
Training species 544.0
Training species 559.0
Training species 581.0
Training species 593.0
Training species 623.0
Training species 643.0
Training species 651.0
Training species 661.0
Training species 694.0
Training species 791.0
Training species 799.0
Training species 822.0
Training species 843.0
Training species 868.0
Training species 890.0
Training species 907.0
Training species 958.0
Training species 963.0
Training species 976.0
Training species 981.0
Training species 1015.0
Training species 1018.0
Training species 1037.0
Training species 1051.0
Training species 1085.0
Training species 1086.0
Trainin

TEST 

In [12]:
test['lon_lat'] = test['lon'] * test['lat']
test['lat2']= test['lat'] ** 2

In [13]:
features = ['lon', 'lat', 'lon_lat', 'year', 'lat2']

X_test = test.groupby('surveyId')[features + ['region', 'country']].first()
X_test = pd.get_dummies(X_test, columns=['region', 'country'])

# aligner avec train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# nettoyage
import numpy as np
X_test = X_test.astype(float)
X_test = X_test.replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0)

In [14]:
preds = {}

for species, model in models.items():
    preds[species] = model.predict_proba(X_test)[:, 1]

preds = pd.DataFrame(preds, index=X_test.index)

In [15]:
preds.head(5)

,53.0,96.0,129.0,146.0,167.0,249.0,254.0,262.0,300.0,340.0,...,10950.0,10969.0,10998.0,11005.0,11054.0,11078.0,11140.0,11157.0,11193.0,11195.0
surveyId,,,,,,,,,,,,,,,,,,,,,
642,0.106074,0.170452,0.100592,0.098761,0.091555,0.107017,0.248736,0.091279,0.094551,0.144243,...,0.093811,0.101560,0.092004,0.157311,0.092545,0.107242,0.197905,0.094119,0.091769,0.188232
1792,0.139590,0.113285,0.091946,0.115524,0.092257,0.107017,0.182553,0.276078,0.094551,0.110405,...,0.092685,0.098103,0.097945,0.189015,0.098607,0.098034,0.192727,0.094119,0.091769,0.147284
3256,0.125323,0.125256,0.091946,0.138482,0.092257,0.107017,0.182553,0.091279,0.093353,0.112611,...,0.097575,0.098103,0.091155,0.096899,0.099119,0.098034,0.146227,0.095163,0.091769,0.147284
3855,0.146258,0.114818,0.091946,0.115524,0.092737,0.107017,0.182553,0.093380,0.094551,0.110405,...,0.098790,0.098103,0.095582,0.189015,0.098607,0.098034,0.185330,0.094119,0.091769,0.147284
4889,0.116566,0.130341,0.103847,0.098761,0.091555,0.107017,0.285196,0.091279,0.094551,0.129853,...,0.093811,0.101540,0.091155,0.139060,0.092545,0.104598,0.175478,0.094119,0.091769,0.191917


In [16]:
K = 25  # valeur recommandée

def get_top_k(row):
    return list(row.sort_values(ascending=False).head(K).index)

predictions = preds.apply(get_top_k, axis=1)

In [17]:
submission = pd.DataFrame({
    'surveyId': predictions.index,
    'predictions': predictions.apply(lambda x: ' '.join(map(str, x)))
})

print(submission)

          surveyId                                        predictions
surveyId                                                             
642            642  4397.0 540.0 5071.0 254.0 4499.0 10317.0 9341....
1792          1792  5557.0 262.0 1254.0 4498.0 1015.0 10600.0 981....
3256          3256  963.0 9647.0 540.0 3722.0 2885.0 1888.0 7999.0...
3855          3855  4498.0 10600.0 540.0 8151.0 10073.0 11005.0 43...
4889          4889  254.0 540.0 10600.0 5071.0 10255.0 4397.0 4499...
...            ...                                                ...
5010108    5010108  5071.0 11140.0 540.0 8151.0 7121.0 981.0 7880....
5010109    5010109  661.0 11140.0 4498.0 6491.0 8151.0 981.0 540.0...
5010110    5010110  661.0 11140.0 4498.0 6491.0 8151.0 981.0 540.0...
5010111    5010111  661.0 11140.0 4498.0 6491.0 8151.0 981.0 540.0...
5010112    5010112  661.0 11140.0 4498.0 6491.0 8151.0 981.0 540.0...

[14784 rows x 2 columns]


In [ ]:
submission.to_csv('submission.csv', index=False)

In [18]:
Y_true = Y.copy()

In [19]:
predictions = preds.apply(get_top_k, axis=1)

In [24]:
print("X index size:", len(X.index))
print("Y index size:", len(Y.index))

print("Intersection:", len(X.index.intersection(Y.index)))

X index size: 87460
Y index size: 87460
Intersection: 87460


In [20]:
Y_pred = pd.DataFrame(0, index=predictions.index, columns=Y.columns)

for idx, species_list in predictions.items():
    for s in species_list:
        if s in Y_pred.columns:
            Y_pred.loc[idx, s] = 1

In [ ]:
Y_true = Y_true.loc[Y_pred.index]

KeyError: "None of [Index([    642,    1792,    3256,    3855,    4889,    5884,    6955,    8956,\n          9037,    9537,\n       ...\n       5010103, 5010104, 5010105, 5010106, 5010107, 5010108, 5010109, 5010110,\n       5010111, 5010112],\n      dtype='int64', name='surveyId', length=14784)] are in the [index]"

In [ ]:
from sklearn.metrics import f1_score

f1_micro = f1_score(Y_true, Y_pred, average='micro')
print("F1-score (micro):", f1_micro)